# Driver Behaviour Analysis using Machine Learning

This notebook develops a machine learning model to classify driving behaviour using sensor data collected from vehicle telemetry. The workflow covers data preprocessing, feature engineering using 5-second sliding windows, model training, evaluation, and exporting the final model for deployment.

### Cell 1 - Import Required Libraries

Import all Python libraries used throughout the project. These include packages for data manipulation, visualization, feature preprocessing, model training, evaluation, and model serialization.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

from xgboost import XGBClassifier

import joblib

### Cell 2 - Load Driver Behaviour Datasets

Load publicly available driver behaviour datasets and combine them into a single DataFrame. This creates a unified dataset for preprocessing and model training.

In [ ]:
DATASETS = [

    "datasets/driving_behavior.csv",

    "datasets/larger_set.csv", 

    "datasets/harsh_driving.csv",

    "datasets/phone_sensor.csv"

]

### Cell 3 -  Clean the Dataset

Remove duplicate entries, handle missing values, and reset the DataFrame index. Cleaning the data ensures that the machine learning model is trained on reliable and consistent samples.

In [ ]:
dfs = []

for file in DATASETS:

    try:

        df = pd.read_csv(file)

        dfs.append(df)

        print(f"Loaded {file} -> {len(df)} rows")

    except Exception as e:

        print(file, e)

data = pd.concat(dfs, ignore_index=True)

print(data.shape)

data.head()

### Cell 4 -  Exploratory Data Analysis

Perform a basic inspection of the dataset by viewing summary statistics, feature distributions, and class balance. This step helps identify data quality issues and understand the characteristics of the collected sensor data.

In [ ]:
data = data.drop_duplicates()

data = data.dropna()

data.reset_index(drop=True, inplace=True)

print(data.shape)

### Cell 5 - Exploratory Data Analysis

Perform a basic inspection of the dataset by viewing summary statistics, feature distributions, and class balance. This step helps identify data quality issues and understand the characteristics of the collected sensor data.

In [ ]:
print(data.info())

print(data.describe())

print(data["label"].value_counts())

### Cell 6 - Feature Engineering using 5-Second Sliding Windows

Instead of classifying individual sensor readings, the data is divided into overlapping 5-second windows. Statistical and derived features are extracted from each window, producing a compact representation of the driver's behaviour over time.

In [ ]:
WINDOW_SIZE = 250      # 5 seconds @ 50 Hz
STEP_SIZE = 125        # 50% overlap

feature_rows = []

for start in range(0, len(data) - WINDOW_SIZE, STEP_SIZE):

    window = data.iloc[start:start + WINDOW_SIZE]

    feature = {}

    # Accelerometer
    for axis in ["ax", "ay", "az"]:
        feature[f"{axis}_mean"] = window[axis].mean()
        feature[f"{axis}_std"] = window[axis].std()
        feature[f"{axis}_max"] = window[axis].max()
        feature[f"{axis}_min"] = window[axis].min()
        feature[f"{axis}_rms"] = np.sqrt(np.mean(window[axis]**2))

    # Gyroscope
    for axis in ["gx", "gy", "gz"]:
        feature[f"{axis}_mean"] = window[axis].mean()
        feature[f"{axis}_std"] = window[axis].std()
        feature[f"{axis}_max"] = window[axis].max()
        feature[f"{axis}_min"] = window[axis].min()
        feature[f"{axis}_rms"] = np.sqrt(np.mean(window[axis]**2))

    # Speed
    if "speed" in window.columns:
        feature["speed_mean"] = window["speed"].mean()
        feature["speed_std"] = window["speed"].std()
        feature["speed_max"] = window["speed"].max()
        feature["speed_min"] = window["speed"].min()

    # Acceleration Magnitude
    acc_mag = np.sqrt(
        window["ax"]**2 +
        window["ay"]**2 +
        window["az"]**2
    )

    feature["acc_mag_mean"] = acc_mag.mean()
    feature["acc_mag_std"] = acc_mag.std()
    feature["acc_mag_max"] = acc_mag.max()

    # Gyroscope Magnitude
    gyro_mag = np.sqrt(
        window["gx"]**2 +
        window["gy"]**2 +
        window["gz"]**2
    )

    feature["gyro_mag_mean"] = gyro_mag.mean()
    feature["gyro_mag_std"] = gyro_mag.std()
    feature["gyro_mag_max"] = gyro_mag.max()

    # Jerk
    jerk = np.diff(acc_mag)

    feature["jerk_mean"] = jerk.mean()
    feature["jerk_std"] = jerk.std()
    feature["jerk_max"] = jerk.max()

    # Majority label inside window
    feature["label"] = window["label"].mode()[0]

    feature_rows.append(feature)

feature_df = pd.DataFrame(feature_rows)

feature_df.head()

### Cell 7 -  Prepare Input Features and Target Labels

Separate the engineered feature set from the target labels. The resulting feature matrix (X) and label vector (y) are used for training and evaluating the machine learning models.

In [ ]:
X = feature_df.drop(columns=["label"])

y = feature_df["label"]

FEATURES = X.columns.tolist()

### Cell - 8 Create the Machine Learning Pipeline

Build a preprocessing and training pipeline using StandardScaler and the selected classifier. Combining preprocessing and classification into a single pipeline ensures identical transformations during both training and inference.

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

### Cell 9 - Train-Test Split

Divide the dataset into training and testing subsets. The training data is used to learn the model parameters, while the testing data provides an unbiased evaluation of model performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

### Train Random Forest Classifier

Train a Random Forest model using the engineered driving features. Random Forest is chosen for its robustness, ability to handle nonlinear relationships, and interpretability through feature importance analysis.

In [ ]:
pipeline.fit(X_train, y_train)

rf_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_pred))

### Train XGBoost Classifier

Train an XGBoost classifier and compare its performance against the Random Forest model. XGBoost often provides higher predictive accuracy by combining multiple weak learners into a powerful ensemble model.

In [ ]:
xgb_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", XGBClassifier(
        random_state=42,
        eval_metric="mlogloss"
    ))
])

xgb_pipeline.fit(X_train, y_train)

xgb_pred = xgb_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, xgb_pred))

### Evaluate Model Performance

Evaluate the trained models using classification metrics such as accuracy, precision, recall, and F1-score. Comparing these metrics helps identify the most suitable model for deployment.

In [ ]:
print("Random Forest")

print(classification_report(y_test, rf_pred))

print("----------------------")

print("XGBoost")

print(classification_report(y_test, xgb_pred))

### Feature Importance Analysis

Visualize the contribution of each engineered feature to the final prediction. Feature importance provides insight into which driving characteristics have the greatest influence on driver behaviour classification.

In [ ]:
importance = pd.Series(
    xgb_pipeline.named_steps["classifier"].feature_importances_,
    index=FEATURES
)

importance.sort_values().plot.barh(figsize=(10,8))

plt.show()

### Confusion Matrix

Generate a confusion matrix to visualize the classification performance across all behaviour classes. This provides a detailed view of correctly and incorrectly classified driving events.

In [ ]:
importance = pd.Series(

    xgb.feature_importances_,

    index=FEATURES

)

importance.sort_values().plot.barh(figsize=(8,5))

plt.show()

## Export the Trained Model

Save the complete preprocessing and classification pipeline as a serialized file. This allows the trained model to be reused for future inference without retraining.

In [ ]:
joblib.dump(xgb_pipeline, "driver_behavior_model.pkl")

print("Model saved.")

### Test Model Inference

Load a sample driving window and perform a prediction using the trained model. This demonstrates how the exported model can classify new, unseen driving behaviour data.

In [ ]:
sample = X_test[0].reshape(1, -1)

prediction = xgb.predict(sample)

print(prediction)

# Conclusion

A complete machine learning pipeline was developed for driver behaviour classification using vehicle sensor data. Raw telemetry was transformed into informative features using 5-second overlapping sliding windows, enabling the models to capture driving patterns rather than individual sensor readings. Among the evaluated classifiers, the best-performing model was exported as a reusable pipeline for future deployment in an embedded driver monitoring system.